# Interactive Exercise: Determine ARIMA Model Parameters

In this exercise, you will use simulated monthly expenditure data to identify reasonable values for the ARIMA parameters \(p\), \(d\), and \(q\).

You will:

1. Select an expenditure category.
2. Determine the differencing order \(d\).
3. Create a stationary series.
4. Examine the ACF and PACF plots.
5. Record a small set of candidate ARIMA models.

> **About the data:** The dataset is simulated for instructional purposes and does not contain official Illinois monthly expenditure figures.


## 1. Load the Required Packages

The code below installs any missing packages and then loads the packages used in this exercise.


In [ ]:
packages <- c(
  "readr",
  "dplyr",
  "ggplot2",
  "forecast",
  "tseries",
  "tibble"
)

installed <- rownames(installed.packages())
missing_packages <- packages[!(packages %in% installed)]

if (length(missing_packages) > 0) {
  install.packages(
    missing_packages,
    repos = "https://cloud.r-project.org"
  )
}

invisible(
  lapply(
    packages,
    library,
    character.only = TRUE
  )
)


## 2. Load the Simulated Monthly Expenditure Data


In [ ]:
df <- read_csv(
  "../data/illinois_monthly_expenditures_simulated.csv"
)

df


## 3. Review the Available Expenditure Series


In [ ]:
df %>%
  distinct(series, label)


## 4. Select an Expenditure Category

Change the value below to examine another category.


In [ ]:
selected_series <- "education"


## 5. Prepare the Monthly Time Series


In [ ]:
plot_df <- df %>%
  filter(series == selected_series) %>%
  mutate(date = as.Date(date)) %>%
  arrange(date)

if (nrow(plot_df) == 0) {
  stop("The selected expenditure series was not found.")
}

start_year <- as.integer(format(min(plot_df$date), "%Y"))
start_month <- as.integer(format(min(plot_df$date), "%m"))

expenditure_ts <- ts(
  plot_df$expenditure,
  start = c(start_year, start_month),
  frequency = 12
)

expenditure_ts


## 6. Determine the Differencing Order \(d\)

The **I** in ARIMA means **integrated**. It records how many times the original series must be differenced before it becomes approximately stationary.

- \(d=0\): use the original series.
- \(d=1\): use the first-differenced series.
- \(d=2\): difference the series twice.

The `ndiffs()` function provides a statistical recommendation for \(d\). Interpret the result together with the plots and ADF test from the previous exercise.


In [ ]:
suggested_d <- forecast::ndiffs(
  expenditure_ts,
  test = "adf"
)

cat(
  "Suggested differencing order: d =",
  suggested_d
)


## 7. Compare the Original and First-Differenced Series


In [ ]:
par(mfrow = c(2, 1))

plot(
  expenditure_ts,
  main = "Original Monthly Expenditure Series",
  ylab = "Expenditure ($ millions)",
  xlab = "Year"
)

plot(
  diff(expenditure_ts),
  main = "First-Differenced Monthly Expenditure Series",
  ylab = "Monthly Change",
  xlab = "Year"
)

par(mfrow = c(1, 1))


## 8. Create the Stationary Series

The ACF and PACF should be examined using the stationary version of the series.


In [ ]:
if (suggested_d == 0) {
  stationary_ts <- expenditure_ts
} else {
  stationary_ts <- diff(
    expenditure_ts,
    differences = suggested_d
  )
}

stationary_ts


## 9. Examine the ACF

The **autocorrelation function (ACF)** measures the relationship between the series and its previous values at different lags.

A clear cutoff in the ACF after lag \(q\) may suggest a moving average term of order \(q\).


In [ ]:
forecast::ggAcf(
  stationary_ts,
  lag.max = 24
) +
  labs(
    title = "Autocorrelation Function",
    x = "Lag",
    y = "Autocorrelation"
  ) +
  theme_minimal()


## 10. Examine the PACF

The **partial autocorrelation function (PACF)** measures the direct relationship at each lag after controlling for shorter lags.

A clear cutoff in the PACF after lag \(p\) may suggest an autoregressive term of order \(p\).


In [ ]:
forecast::ggPacf(
  stationary_ts,
  lag.max = 24
) +
  labs(
    title = "Partial Autocorrelation Function",
    x = "Lag",
    y = "Partial Autocorrelation"
  ) +
  theme_minimal()


## 11. Record Candidate ARIMA Models

Use the ACF and PACF plots to identify a small set of reasonable candidate models.

The examples below can be edited.


In [ ]:
candidate_models <- tibble(
  p = c(1, 0, 1),
  d = c(suggested_d, suggested_d, suggested_d),
  q = c(0, 1, 1),
  model = paste0(
    "ARIMA(",
    p, ",",
    d, ",",
    q, ")"
  ),
  reason = c(
    "PACF suggests one autoregressive term",
    "ACF suggests one moving average term",
    "Both plots suggest a mixed model"
  )
)

candidate_models


## Questions for Reflection

1. What value of \(d\) was recommended?

2. Does the original series appear to contain a trend?

3. What does the ACF suggest about \(q\)?

4. What does the PACF suggest about \(p\)?

5. Which candidate ARIMA models should be evaluated next?

6. Are any seasonal spikes visible at lags 12 or 24?


## Try Another Expenditure Category

Return to the `selected_series` cell, choose another expenditure category, and rerun the notebook.

Compare whether the suggested values of \(p\), \(d\), and \(q\) change across categories.


# Next Step

In the next exercise, you will estimate the candidate models, compare information criteria, review residual diagnostics, and select the final ARIMA specification.
